In [3]:
import pandas as pd
from sklearn.model_selection import train_test_split
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from sklearn.preprocessing import StandardScaler

In [4]:
if torch.cuda.is_available():
    print("GPU available")
    device = torch.device("cuda")
else:
    device = torch.device("cpu")

GPU available


In [5]:
torch.manual_seed(42)

In [6]:
from torchvision import datasets, transforms

transform = transforms.ToTensor()

train_dataset = datasets.FashionMNIST(root='./data', train=True, download=True, transform=transform)
test_dataset = datasets.FashionMNIST(root='./data', train=False, download=True, transform=transform)

100%|██████████| 26.4M/26.4M [00:00<00:00, 114MB/s]
100%|██████████| 29.5k/29.5k [00:00<00:00, 3.74MB/s]
100%|██████████| 4.42M/4.42M [00:00<00:00, 55.3MB/s]
100%|██████████| 5.15k/5.15k [00:00<00:00, 29.0MB/s]


In [7]:
train_dataset

Dataset FashionMNIST
    Number of datapoints: 60000
    Root location: ./data
    Split: Train
    StandardTransform
Transform: ToTensor()

In [8]:
class MyNN(nn.Module):
    def __init__(self, input_size, output_size, no_of_layers, neurons_list, dropout_rate):
        super().__init__()

        layers = []

        for i in range(no_of_layers):
            if i==0:
                layers.append(nn.Linear(input_size, neurons_list[0]))
                layers.append(nn.BatchNorm1d(neurons_list[0]))
                layers.append(nn.ReLU())
                layers.append(nn.Dropout(dropout_rate))
            else:
                layers.append(nn.Linear(neurons_list[i-1], neurons_list[i]))
                layers.append(nn.BatchNorm1d(neurons_list[i]))
                layers.append(nn.ReLU())
                layers.append(nn.Dropout(dropout_rate))

        layers.append(nn.Linear(neurons_list[-1], output_size))

        self.network = nn.Sequential(*layers)

    def forward(self, x):
        x = x.view(x.size(0), -1) # Flatten the input
        return self.network(x)

In [9]:
def objective(trial):

    no_of_layers = trial.suggest_int("no_of_layers", 1, 5)
    neurons_list = []
    for i in range(no_of_layers):
        neurons_list.append(trial.suggest_int(f'neuron_{i}',16, 128, step=8))

    epochs = trial.suggest_int("epochs", 10, 50, step=10)
    batch_size = trial.suggest_categorical("batch_size", [16, 32, 64])
    learning_rate = trial.suggest_float("learning_rate", 1e-5, 1e-1, log=True)
    dropout_rate = trial.suggest_float("dropout_rate", 0.1, 0.5, step=0.1)
    optimizer_name = trial.suggest_categorical("optimizer", ['Adam', 'SGD', 'RMSprop'])
    weight_decay = trial.suggest_float("weight_decay", 1e-5, 1e-3, log=True)

    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
    test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

    model = MyNN(784, 10, no_of_layers, neurons_list, dropout_rate)
    model.to(device)

    criterion = nn.CrossEntropyLoss()
    if optimizer_name == 'Adam':
        optimizer = optim.Adam(model.parameters(), lr=learning_rate, weight_decay=weight_decay)
    elif optimizer_name == 'SGD':
        optimizer = optim.SGD(model.parameters(), lr=learning_rate, weight_decay=weight_decay)
    else:
        optimizer = optim.RMSprop(model.parameters(), lr=learning_rate, weight_decay=weight_decay)


    for epoch in range(epochs):

        for batch_data, batch_label in train_loader:
            batch_data = batch_data.to(device)
            batch_label = batch_label.to(device)

            out = model(batch_data)
            loss = criterion(out, batch_label)

            optimizer.zero_grad()
            loss.backward()

            optimizer.step()


    model.eval()
    total = 0
    correct = 0
    with torch.no_grad():
        for batch_data, batch_label in test_loader:
            batch_data = batch_data.to(device)
            batch_label = batch_label.to(device)

            out = model(batch_data)
            _, predicted = torch.max(out.data, 1)
            total += batch_label.size(0)
            correct += (predicted == batch_label).sum().item()

    accuracy = 100 * correct / total
    return accuracy

In [10]:
!pip install optuna

In [12]:
import optuna

study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=10)

[I 2025-10-29 22:14:29,322] A new study created in memory with name: no-name-9d50a2c5-e256-4bb2-bb24-21ba8a7674d6
[I 2025-10-29 22:16:00,136] Trial 0 finished with value: 69.88 and parameters: {'no_of_layers': 4, 'neuron_0': 40, 'neuron_1': 32, 'neuron_2': 80, 'neuron_3': 16, 'epochs': 10, 'batch_size': 64, 'learning_rate': 0.043718805059305234, 'dropout_rate': 0.1, 'optimizer': 'RMSprop', 'weight_decay': 0.00021713105705983103}. Best is trial 0 with value: 69.88.
[I 2025-10-29 22:17:31,475] Trial 1 finished with value: 52.42 and parameters: {'no_of_layers': 5, 'neuron_0': 16, 'neuron_1': 16, 'neuron_2': 32, 'neuron_3': 96, 'neuron_4': 96, 'epochs': 10, 'batch_size': 64, 'learning_rate': 0.0489136741784657, 'dropout_rate': 0.2, 'optimizer': 'RMSprop', 'weight_decay': 7.745652547058748e-05}. Best is trial 0 with value: 69.88.
[I 2025-10-29 22:21:13,824] Trial 2 finished with value: 69.55 and parameters: {'no_of_layers': 4, 'neuron_0': 112, 'neuron_1': 24, 'neuron_2': 24, 'neuron_3': 32,

KeyboardInterrupt: 

In [13]:
print(study.best_value)
print(study.best_params)

84.3
{'no_of_layers': 3, 'neuron_0': 96, 'neuron_1': 16, 'neuron_2': 64, 'epochs': 50, 'batch_size': 16, 'learning_rate': 1.352897077138141e-05, 'dropout_rate': 0.4, 'optimizer': 'Adam', 'weight_decay': 1.2211012803970173e-05}
